# Video-LLaVA / MuLER POPE Benchmark Evaluation (Google Colab)

Evaluates and compares your Stage 1 training progression across **3 models** (plus an optional 4th reference arm) on the **POPE** (Polling-based Object Probing Evaluation) hallucination benchmark:

1. **Untrained (Step 0)** — `lmsys/vicuna-7b-v1.5` + `LanguageBind_Image` + randomly initialized `mm_projector` + zeroed Gate v2 (no training)
2. **`final` (Stage 1, Step 867)** — ~7.5% of Epoch 1
3. **`checkpoint-4750` (Stage 1, Step 4750)** — ~41% of Epoch 1
4. *(Optional)* **Official `LanguageBind/Video-LLaVA-7B` (Stage 2 Baseline)** — full Stage 2 instruction-tuned upper-bound reference

### What POPE Measures
- **3 Splits** (`random`, `popular`, `adversarial`), 3,000 yes/no questions each (**9,000 total**) across **500 COCO `val2014` images** (~80 MB download).
- **Metrics**: Accuracy, Precision, Recall, F1 Score, and Yes-Ratio (detects "yes" bias / object hallucination).

### Estimated Runtime (9,000 questions per model)
| Colab GPU | Auto Precision | Time per Model | 3 Models |
|---|---|---|---|
| **A100 (40GB/80GB)** | `bf16` | ~6–10 min | ~20–30 min |
| **L4 (24GB)** | `bf16` | ~12–18 min | ~35–55 min |
| **T4 (15GB)** | `4bit-fp16` | ~25–40 min | ~75–120 min |

*(All evaluations are resumable — if Colab disconnects, re-running a cell resumes from the last flushed question.)*

## 1. Check GPU & Auto-Selected Precision

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    prec = '4bit-fp16' if vram < 20 else ('bf16' if torch.cuda.is_bf16_supported() else 'fp16')
    print(f"GPU: {name} | VRAM: {vram:.1f} GB | Auto precision: {prec}")
else:
    print("WARNING: No CUDA GPU detected! Switch Runtime -> Change runtime type -> GPU.")

## 2. Mount Google Drive (Optional but Recommended)
Used to read your checkpoints from Drive if you added a shortcut at `/content/drive/MyDrive/Video-LLaVA` (otherwise the script downloads `final` and `checkpoint-4750` directly from their Drive IDs) and to persist all POPE reports + raw answers back to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_EVAL_OUT = "/content/drive/MyDrive/Video-LLaVA/eval_results/pope"
os.makedirs(DRIVE_EVAL_OUT, exist_ok=True)
print(f"Results will be synced to: {DRIVE_EVAL_OUT}")

## 3. Clone Repository & Install Dependencies
> **Note:** Make sure your latest commit containing `scripts/eval_pope_colab.py` and the loader fixes is pushed to `https://github.com/davidrimon2004/grad_project` (or upload the updated repo files to `/content/grad_project`).

In [ ]:
%cd /content
![ ! -d "grad_project" ] && git clone https://github.com/davidrimon2004/grad_project
%cd /content/grad_project
!git pull

!pip install -q --upgrade pip
!pip install -q transformers tokenizers sentencepiece shortuuid accelerate peft bitsandbytes einops einops-exts timm huggingface_hub gdown matplotlib tqdm
!pip install -q -e .

## 4. Prepare POPE Benchmark Data (~80 MB)
Downloads the 3 official POPE JSON splits (`adversarial`, `popular`, `random`) and fetches only the **500 unique COCO `val2014` images** referenced by POPE.

In [ ]:
!python scripts/eval_pope_colab.py \
    --action prepare \
    --eval_dir /content/eval/pope

## 5. Fetch & Inspect Checkpoints (`final`, `checkpoint-4750`, and `untrained_step0`)
Downloads `final` (step 867) and `checkpoint-4750` (step 4,750) to local SSD, prepares `untrained_step0` (config-only, no `mm_projector.bin`), and prints Gate v2 (`early_fusion_gate_linear`) norms + training loss stats for both checkpoints.

In [ ]:
!python scripts/eval_pope_colab.py \
    --action download_ckpt \
    --ckpt_name all \
    --drive_ckpt_root /content/drive/MyDrive/Video-LLaVA/checkpoints/videollava-7b-pretrain \
    --local_ckpt_root /content/checkpoints/videollava-7b-pretrain

!python scripts/eval_pope_colab.py \
    --action inspect_ckpt \
    --ckpt_name all \
    --local_ckpt_root /content/checkpoints/videollava-7b-pretrain

## 6. Optional Quick 3-Way Smoke Test (50 Samples / Split = 150 Questions Each, ~3 min)
Run this first to verify all 3 models (`Untrained Step 0`, `final`, `checkpoint-4750`) end-to-end and preview a fast 3-way table before running all 9,000 questions.

In [ ]:
!python scripts/eval_pope_colab.py --action eval --untrained \
    --model_path /content/checkpoints/videollava-7b-pretrain/untrained_step0 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/pope \
    --answers_file /content/eval/pope/answers/smoke_untrained.jsonl \
    --metrics_file /content/eval/pope/results/smoke_untrained_metrics.json \
    --max_samples_per_split 50 --precision auto --label "Untrained (Step 0)"

!python scripts/eval_pope_colab.py --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/final \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/pope \
    --answers_file /content/eval/pope/answers/smoke_final.jsonl \
    --metrics_file /content/eval/pope/results/smoke_final_metrics.json \
    --max_samples_per_split 50 --precision auto --label "final (Stage 1, step 867)"

!python scripts/eval_pope_colab.py --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/checkpoint-4750 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/pope \
    --answers_file /content/eval/pope/answers/smoke_4750.jsonl \
    --metrics_file /content/eval/pope/results/smoke_4750_metrics.json \
    --max_samples_per_split 50 --precision auto --label "checkpoint-4750 (Stage 1, step 4750)"

!python scripts/eval_pope_colab.py --action compare \
    --metrics_files \
      /content/eval/pope/results/smoke_untrained_metrics.json \
      /content/eval/pope/results/smoke_final_metrics.json \
      /content/eval/pope/results/smoke_4750_metrics.json \
    --output_dir /content/eval/pope/results/smoke

## 7A. Full POPE Evaluation: Untrained (Step 0, No Training)

In [ ]:
!python scripts/eval_pope_colab.py \
    --action eval \
    --untrained \
    --model_path /content/checkpoints/videollava-7b-pretrain/untrained_step0 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/pope \
    --answers_file /content/eval/pope/answers/untrained_step0.jsonl \
    --metrics_file /content/eval/pope/results/untrained_step0_metrics.json \
    --precision auto \
    --label "Untrained (Step 0)"

## 7B. Full POPE Evaluation: `final` (Stage 1, Step 867)

In [ ]:
!python scripts/eval_pope_colab.py \
    --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/final \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/pope \
    --answers_file /content/eval/pope/answers/final_stage1_step867.jsonl \
    --metrics_file /content/eval/pope/results/final_metrics.json \
    --precision auto \
    --label "final (Stage 1, step 867)"

## 7C. Full POPE Evaluation: `checkpoint-4750` (Stage 1, Step 4750)

In [ ]:
!python scripts/eval_pope_colab.py \
    --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/checkpoint-4750 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/pope \
    --answers_file /content/eval/pope/answers/checkpoint_4750_stage1.jsonl \
    --metrics_file /content/eval/pope/results/checkpoint_4750_metrics.json \
    --precision auto \
    --label "checkpoint-4750 (Stage 1, step 4750)"

## 8. (Optional) Official `LanguageBind/Video-LLaVA-7B` Stage 2 Baseline
Run this cell if you also want the fully instruction-tuned Video-LLaVA-7B baseline included in the comparison table and chart.

In [ ]:
!python scripts/eval_pope_colab.py \
    --action eval \
    --model_path LanguageBind/Video-LLaVA-7B \
    --eval_dir /content/eval/pope \
    --answers_file /content/eval/pope/answers/videollava_7b_baseline.jsonl \
    --metrics_file /content/eval/pope/results/baseline_metrics.json \
    --precision auto \
    --label "Official Video-LLaVA-7B (Stage 2 Baseline)"

## 9. Multi-Model Comparison Table, Chart & Drive Sync
Automatically compares all evaluated models (`Untrained`, `final`, `checkpoint-4750`, and `Official Video-LLaVA-7B` if Section 8 was run), plots F1 / Accuracy / Yes-Ratio, and syncs everything to Google Drive.

In [ ]:
import os, json, shutil, subprocess
import matplotlib.pyplot as plt
import numpy as np

candidate_files = [
    '/content/eval/pope/results/untrained_step0_metrics.json',
    '/content/eval/pope/results/final_metrics.json',
    '/content/eval/pope/results/checkpoint_4750_metrics.json',
    '/content/eval/pope/results/baseline_metrics.json',
]
existing_files = [p for p in candidate_files if os.path.exists(p)]
assert len(existing_files) >= 2, f'Expected at least 2 metrics files, found: {existing_files}'

cmd = [
    'python', 'scripts/eval_pope_colab.py', '--action', 'compare',
    '--metrics_files', *existing_files,
    '--output_dir', '/content/eval/pope/results',
    '--drive_output_dir', '/content/drive/MyDrive/Video-LLaVA/eval_results/pope',
]
subprocess.run(cmd, check=True)

with open('/content/eval/pope/results/pope_comparison_results.json') as f:
    data = json.load(f)

models = data['models']
splits = ['random', 'popular', 'adversarial']
metrics = ['f1', 'accuracy', 'yes_ratio']
titles = ['F1 Score (%)', 'Accuracy (%)', 'Yes Ratio (%)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(len(splits))
n_models = len(models)
width = min(0.8 / max(n_models, 1), 0.25)

for ax, m_key, title in zip(axes, metrics, titles):
    for idx, mdl in enumerate(models):
        offset = (idx - (n_models - 1) / 2) * width
        vals = [mdl['splits'][s][m_key] * 100 for s in splits]
        bars = ax.bar(x + offset, vals, width, label=mdl['name'])
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f'{h:.1f}', (bar.get_x() + bar.get_width()/2, h), ha='center', va='bottom', fontsize=7)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([s.capitalize() for s in splits])
    ax.set_ylim(0, 108)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

axes[0].legend(loc='lower left', fontsize=7)
plt.tight_layout()
chart_path = '/content/eval/pope/results/pope_comparison_chart.png'
plt.savefig(chart_path, dpi=150)
plt.show()

if os.path.exists('/content/drive/MyDrive/Video-LLaVA/eval_results/pope'):
    shutil.copy2(chart_path, '/content/drive/MyDrive/Video-LLaVA/eval_results/pope/pope_comparison_chart.png')
    print('Chart saved to Google Drive.')